In [1]:
!wget https://people.sc.fsu.edu/~jburkardt/datasets/tsp/att48.tsp
!wget https://people.sc.fsu.edu/~jburkardt/datasets/tsp/five_d.txt
!wget https://people.sc.fsu.edu/~jburkardt/datasets/tsp/gr17.tsp
!wget https://people.sc.fsu.edu/~jburkardt/datasets/tsp/p01.tsp

--2025-01-30 19:01:06--  https://people.sc.fsu.edu/~jburkardt/datasets/tsp/att48.tsp
Resolving people.sc.fsu.edu (people.sc.fsu.edu)... 144.174.0.22
Connecting to people.sc.fsu.edu (people.sc.fsu.edu)|144.174.0.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889 [text/plain]
Saving to: ‘att48.tsp’

att48.tsp           100%[===================>]     889  --.-KB/s    in 0s      

2025-01-30 19:01:06 (346 MB/s) - ‘att48.tsp’ saved [889/889]

--2025-01-30 19:01:07--  https://people.sc.fsu.edu/~jburkardt/datasets/tsp/five_d.txt
Resolving people.sc.fsu.edu (people.sc.fsu.edu)... 144.174.0.22
Connecting to people.sc.fsu.edu (people.sc.fsu.edu)|144.174.0.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 121 [text/plain]
Saving to: ‘five_d.txt’

five_d.txt          100%[===================>]     121  --.-KB/s    in 0s      

2025-01-30 19:01:07 (49.8 MB/s) - ‘five_d.txt’ saved [121/121]

--2025-01-30 19:01:07--  https://people.sc.fsu.edu/~

In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt
import time
import random
from itertools import permutations

# 讀取數據集
- FIVE is a set of **5** cities. The minimal tour has length 19.
https://people.sc.fsu.edu/~jburkardt/datasets/tsp/tsp.html#:~:text=FIVE%20is%20a%20set%20of%205%20cities.%20The%20minimal%20tour%20has%20length%2019.

- P01 is a set of **15** cities. It is NOT from TSPLIB. The minimal cost is 291.
https://people.sc.fsu.edu/~jburkardt/datasets/tsp/tsp.html#:~:text=P01%20is%20a%20set%20of%2015%20cities.%20It%20is%20NOT%20from%20TSPLIB.%20The%20minimal%20cost%20is%20291.

- GR17 is a set of 17 cities, from TSPLIB. The minimal tour has length 2085.
https://people.sc.fsu.edu/~jburkardt/datasets/tsp/tsp.html#:~:text=GR17%20is%20a%20set%20of%2017%20cities%2C%20from%20TSPLIB.%20The%20minimal%20tour%20has%20length%202085.

In [3]:
import numpy as np

def build_symmetric_matrix(dimension, edge_weights):
    matrix = np.zeros((dimension, dimension), dtype=int)
    index = 0

    for i in range(dimension):
        for j in range(i + 1):  # Only fill the lower triangle
            matrix[i, j] = edge_weights[index]
            matrix[j, i] = edge_weights[index]  # Symmetric fill
            index += 1

    return matrix

def check_pure_digit(file_content):
    lines = [line.rstrip("\n") for line in file_content]
    matrix = []
    for line in file_content:
        line = line.rstrip('\n')
        if len(line)>1:
            try:
                matrix.append(list(map(float, line.split())))
            except:
                return None, None, None, False

    # 確認矩陣是方陣，且對角線元素為 0
    dim = len(matrix)  # 矩陣的大小
    for i in range(dim):
        if len(matrix[i]) != dim:  # 確保每行有相同數量的元素
            return None, None, None,False
        if matrix[i][i] != 0:  # 檢查對角線元素是否為 0
            return None, None, None, False

    return dim, "FULL_MATRIX", matrix, True

In [4]:
def read_data(file_name):
    dimension = None
    edge_weight_format = None
    dis_matrix = []
    reading_weights = False

    with open(file_name, 'r') as f:
        lines = f.readlines()
        dimension, edge_weight_format, dis_matrix, check_res = check_pure_digit(lines)
        
        if check_res:
            return dimension, edge_weight_format, np.array(dis_matrix)
        else:
            dis_matrix = []
            # Initial processing of the file
            for line in lines:
                line = line.strip()

                if line.startswith("DIMENSION"):
                    dimension = int(line.split(":")[1].strip())
#                     print(f"Dimension: {dimension}")

                elif line.startswith("EDGE_WEIGHT_FORMAT"):
                    edge_weight_format = line.split(":")[1].strip()

                elif line.startswith("EDGE_WEIGHT_SECTION"):
                    # Start reading the edge weight section
                    reading_weights = True
                    continue

                elif line.startswith("EOF"):
                    # End of the section
                    break

                # Process the edge weights after the "EDGE_WEIGHT_SECTION"
                if reading_weights:
                    try:
                        dis_matrix.append(list(map(int, line.split())))
                    except ValueError:
                        print(f"Skipping invalid line: {line}")
                        
            if edge_weight_format == "FULL_MATRIX" and dis_matrix:
                dis_matrix = np.array(dis_matrix)
                return dimension, edge_weight_format, dis_matrix
            
            elif edge_weight_format == "LOWER_DIAG_ROW":
                edge_weights = []
                for weight_line in dis_matrix:
                    edge_weights.extend(weight_line)

                dis_matrix = build_symmetric_matrix(dimension, edge_weights)
                return dimension, edge_weight_format, np.array(dis_matrix)

    return None, None, None  # Return None if parsing fails




In [5]:
# Test the function with your file path
file_path = "p01.tsp"
p01_dimension, p01_edge_weight_format, p01_edge_weights = read_data(file_path)
print(f"Dimension: {p01_dimension}")
print(f"Edge weight format: {p01_edge_weight_format}")
print(f"Edge weights matrix:\n{p01_edge_weights}")

Dimension: 15
Edge weight format: FULL_MATRIX
Edge weights matrix:
[[ 0 29 82 46 68 52 72 42 51 55 29 74 23 72 46]
 [29  0 55 46 42 43 43 23 23 31 41 51 11 52 21]
 [82 55  0 68 46 55 23 43 41 29 79 21 64 31 51]
 [46 46 68  0 82 15 72 31 62 42 21 51 51 43 64]
 [68 42 46 82  0 74 23 52 21 46 82 58 46 65 23]
 [52 43 55 15 74  0 61 23 55 31 33 37 51 29 59]
 [72 43 23 72 23 61  0 42 23 31 77 37 51 46 33]
 [42 23 43 31 52 23 42  0 33 15 37 33 33 31 37]
 [51 23 41 62 21 55 23 33  0 29 62 46 29 51 11]
 [55 31 29 42 46 31 31 15 29  0 51 21 41 23 37]
 [29 41 79 21 82 33 77 37 62 51  0 65 42 59 61]
 [74 51 21 51 58 37 37 33 46 21 65  0 61 11 55]
 [23 11 64 51 46 51 51 33 29 41 42 61  0 62 23]
 [72 52 31 43 65 29 46 31 51 23 59 11 62  0 59]
 [46 21 51 64 23 59 33 37 11 37 61 55 23 59  0]]


In [6]:
# Test the function with your file path
file_path = "gr17.tsp"
gr17_dimension, gr17_edge_weight_format, gr17_edge_weights = read_data(file_path)
print(f"Dimension: {gr17_dimension}")
print(f"Edge weight format: {gr17_edge_weight_format}")
print(f"Edge weights matrix:\n{gr17_edge_weights}")

Dimension: 17
Edge weight format: LOWER_DIAG_ROW
Edge weights matrix:
[[  0 633 257  91 412 150  80 134 259 505 353 324  70 211 268 246 121]
 [633   0 390 661 227 488 572 530 555 289 282 638 567 466 420 745 518]
 [257 390   0 228 169 112 196 154 372 262 110 437 191  74  53 472 142]
 [ 91 661 228   0 383 120  77 105 175 476 324 240  27 182 239 237  84]
 [412 227 169 383   0 267 351 309 338 196  61 421 346 243 199 528 297]
 [150 488 112 120 267   0  63  34 264 360 208 329  83 105 123 364  35]
 [ 80 572 196  77 351  63   0  29 232 444 292 297  47 150 207 332  29]
 [134 530 154 105 309  34  29   0 249 402 250 314  68 108 165 349  36]
 [259 555 372 175 338 264 232 249   0 495 352  95 189 326 383 202 236]
 [505 289 262 476 196 360 444 402 495   0 154 578 439 336 240 685 390]
 [353 282 110 324  61 208 292 250 352 154   0 435 287 184 140 542 238]
 [324 638 437 240 421 329 297 314  95 578 435   0 254 391 448 157 301]
 [ 70 567 191  27 346  83  47  68 189 439 287 254   0 145 202 289  55]
 [211 4

In [7]:
# Test the function with your file path
file_path = "five_d.txt"
five_dimension, five_edge_weight_format, five_edge_weights = read_data(file_path)
print(f"Dimension: {five_dimension}")
print(f"Edge weight format: {five_edge_weight_format}")
print(f"Edge weights matrix:\n{five_edge_weights}")

Dimension: 5
Edge weight format: FULL_MATRIX
Edge weights matrix:
[[0. 3. 4. 2. 7.]
 [3. 0. 4. 6. 3.]
 [4. 4. 0. 5. 8.]
 [2. 6. 5. 0. 6.]
 [7. 3. 8. 6. 0.]]


# Genetic algorithm

In [8]:
import numpy as np
import random

# 設置隨機種子，確保結果可復現
random.seed(42)
np.random.seed(42)


# ---------------------- 遺傳算法函數 ----------------------

# 初始化種群（隨機排列城市編號）
def init_population(size, num_cities):
    return [random.sample(range(num_cities), num_cities) for _ in range(size)]

# 計算適應度（路徑總距離的倒數）
def fitness_custom(route, distance_matrix):
    return 1.0 / sum(distance_matrix[route[i], route[i + 1]] for i in range(len(route) - 1)) + 1 / distance_matrix[route[-1], route[0]]

# 選擇（輪盤賭法）
def selection(population, fitness_function, distance_matrix):
    fitness_scores = [fitness_function(route, distance_matrix) for route in population]
    total_fitness = sum(fitness_scores)
    probabilities = [f / total_fitness for f in fitness_scores]
    return population[np.random.choice(len(population), p=probabilities)]

# 交叉（部分映射交叉 PMX）
def crossover(parent1, parent2):
    size = len(parent1)
    cxpoint1, cxpoint2 = sorted(random.sample(range(size), 2))
    child = [-1] * size
    child[cxpoint1:cxpoint2] = parent1[cxpoint1:cxpoint2]

    mapping = {parent1[i]: parent2[i] for i in range(cxpoint1, cxpoint2)}
    for i in range(size):
        if i < cxpoint1 or i >= cxpoint2:
            val = parent2[i]
            while val in mapping:
                val = mapping[val]
            child[i] = val
    return child

# 變異（兩個隨機位置交換）
def mutate(route, mutation_rate):
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]
    return route

# 遺傳算法主循環
def genetic_algorithm(edge_weights, population_size = 100, mut_rate = 0.1, generations = 1000):
    num_cities = len(edge_weights)

    population = init_population(population_size, num_cities)
    
    for _ in range(generations):
        new_population = []
        for _ in range(population_size // 2):
            parent1, parent2 = selection(population, fitness_custom, edge_weights), selection(population, fitness_custom, edge_weights)
            child1, child2 = crossover(parent1, parent2), crossover(parent2, parent1)
            new_population.extend([mutate(child1, mut_rate), mutate(child2, mut_rate)])
        population = new_population

    best_route = min(population, key=lambda route: sum(edge_weights[route[i], route[i+1]] for i in range(len(route) - 1)))
    best_distance = sum(edge_weights[best_route[i], best_route[i+1]] for i in range(len(best_route) - 1)) + edge_weights[best_route[-1], best_route[0]]
    return best_route, best_distance




In [9]:
# 運行 GA
best_route, best_distance = genetic_algorithm(five_edge_weights)
print("最佳路徑:", best_route)
print("最短距離:", best_distance)

最佳路徑: [3, 2, 0, 1, 4]
最短距離: 21.0


In [10]:
# 運行 GA
best_route, best_distance = genetic_algorithm(gr17_edge_weights)
print("最佳路徑:", best_route)
print("最短距離:", best_distance)

最佳路徑: [12, 10, 2, 9, 1, 6, 0, 7, 16, 8, 11, 15, 13, 4, 14, 5, 3]
最短距離: 3396


In [11]:
# 運行 GA
best_route, best_distance = genetic_algorithm(p01_edge_weights)
print("最佳路徑:", best_route)
print("最短距離:", best_distance)

最佳路徑: [8, 6, 10, 3, 5, 13, 11, 1, 0, 12, 4, 2, 9, 7, 14]
最短距離: 463


### 使用 NetworkX

In [12]:
import numpy as np
import networkx as nx
from networkx.algorithms.approximation import traveling_salesman_problem


# 建立 NetworkX 圖

def create_networkx_graph(edge_weights):
    G = nx.complete_graph(len(edge_weights))
    for i in range(len(edge_weights)):
        for j in range(len(edge_weights)):
            if i != j:
                G[i][j]['weight'] = edge_weights[i, j]
                
    return G

Gp01 = create_networkx_graph(p01_edge_weights)
Ggr17 = create_networkx_graph(gr17_edge_weights)
Gfive_d = create_networkx_graph(five_edge_weights)


# 使用最小生成樹近似 TSP
tsp_route_p01 = traveling_salesman_problem(Gp01, cycle=True)
tsp_route_gr17 = traveling_salesman_problem(Ggr17, cycle=True)
tsp_route_five_d = traveling_salesman_problem(Gfive_d, cycle=True)

print("p01.tsp 的近似 TSP 路徑:", tsp_route_p01)
print("gr17.tsp 的近似 TSP 路徑:", tsp_route_gr17)
print("five_d.txt 的近似 TSP 路徑:", tsp_route_five_d)


p01.tsp 的近似 TSP 路徑: [0, 10, 3, 5, 7, 1, 9, 11, 13, 2, 6, 4, 8, 14, 12, 0]
gr17.tsp 的近似 TSP 路徑: [0, 15, 11, 8, 3, 12, 6, 16, 13, 14, 2, 10, 9, 1, 4, 5, 7, 6, 0]
five_d.txt 的近似 TSP 路徑: [0, 2, 1, 4, 1, 0, 3, 0]
